# Phase-1 one-click experiment notebook

Single parameterized entry point for the 2×2 Phase-1 experiment matrix
(7 games × {PPO, ACH} × {mirror, league}). Trains one cell (or all of them),
runs the full eval toolkit, and renders the payoff / forgetting /
non-transitivity plots.

Top-to-bottom readable; rerunnable from a clean kernel (AGENTS.md §7).
Imports from `mjai.*`; does not reimplement logic.

In [ ]:
# Parameters — edit these to run a single cell of the matrix, or set
# RUN_ALL_MATRIX=True to sweep all 28 cells.
GAME          = 'kuhn'      # one of: brps, kuhn, leduc, liars_dice1, goofspiel5_ii, oshi_zumo, ttt
ALGO          = 'ach'       # 'ppo' | 'ach'
SELF_PLAY     = 'mirror'    # 'mirror' | 'league'
N_STEPS       = 1000
RUN_ALL_MATRIX = False      # True => sweep all 7 games × 2 algos × 2 modes

import sys, pathlib; sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))
from mjai.scripts.experiment import ExperimentConfig, run_experiment
from mjai.scripts.evaluate import _load_policy
from mjai.agents.ckpt_io import discover_checkpoints
from mjai.config.game_config import load_all_game_configs
print('Available games:', sorted(load_all_game_configs().keys()))

In [ ]:
# Build the list of experiment configs to run.
GAMES = ['brps', 'kuhn', 'leduc', 'liars_dice1', 'goofspiel5_ii', 'oshi_zumo', 'ttt']
ALGOS = ['ppo', 'ach']
MODES = ['mirror', 'league']

if RUN_ALL_MATRIX:
    cells = [(g, a, m) for g in GAMES for a in ALGOS for m in MODES]
else:
    cells = [(GAME, ALGO, SELF_PLAY)]
print(f'Will run {len(cells)} experiment(s).')

In [ ]:
# Train.
run_dirs = {}
for game, algo, mode in cells:
    cfg = ExperimentConfig(
        game=game, algo=algo, self_play_mode=mode,
        policy_kind='tabular', n_steps=N_STEPS,
        save_every_steps=max(N_STEPS // 5, 50),
        out_dir=f'runs/{game}_{algo}_{mode}', seed=0,
    )
    print(f'
=== {game}/{algo}/{mode} ===')
    run_dirs[(game, algo, mode)] = run_experiment(cfg)
print('
All training complete.')

In [ ]:
# Evaluate each run: equilibrium metric + cross-play matrix across checkpoints.
from mjai.eval.nash import evaluate_equilibrium
from mjai.eval.crossplay import cross_play_matrix, worst_case_win_rate, forgetting_metric, nontransitivity_score
from mjai.games.loader import load_game
from mjai.pipeline.rollout import RolloutConfig, RolloutWorkerCore

rows = []
for (game, algo, mode), run_dir in run_dirs.items():
    spec = load_game(game)
    ckpts = discover_checkpoints(run_dir / 'checkpoints')
    if not ckpts: continue
    final_p, _ = _load_policy(ckpts[-1][0])
    eq = evaluate_equilibrium(spec, final_p)
    policies = []
    for cdir, _ in ckpts:
        try: policies.append(_load_policy(cdir)[0])
        except Exception: pass
    if len(policies) >= 2:
        runner = RolloutWorkerCore(spec, learner_player=0, config=RolloutConfig(n_episodes=30, seed=0))
        cpr = cross_play_matrix(spec, policies, runner, n_episodes=30)
        wc = worst_case_win_rate(cpr)
        fg = forgetting_metric(cpr, early_indices=list(range(1, len(policies))))
        nt = nontransitivity_score(cpr)
    else:
        wc = fg = nt = float('nan')
    primary = eq.get('exploitability', eq.get('nash_conv', eq.get('exact_nash_distance', float('nan'))))
    rows.append({'game': game, 'algo': algo, 'mode': mode, 'primary_metric': primary, **eq,
                 'worst_case_wr': wc, 'forgetting': fg, 'nontransitivity': nt})
    print(f'{game:14s} {algo:4s} {mode:7s} primary={primary:.4g} worst_wr={wc:.3f} forget={fg:.3f} nontrans={nt:.3f}')

In [ ]:
# Render the comparison table + plots.
import pandas as pd
df = pd.DataFrame(rows)
df

In [ ]:
# Headline comparison: ACH vs PPO, mirror vs league, per game.
if len(df) >= 4:
    pivot = df.pivot_table(index='game', columns=['algo', 'mode'], values='primary_metric')
    print('Primary metric (lower = closer to Nash) by cell:')
    display(pivot)  # noqa
else:
    print('Run more cells (RUN_ALL_MATRIX=True) to see the full pivot.')

## Interpretation

- **primary_metric** (exploitability / NashConv / exact-Nash distance, lower = closer to Nash).
- **forgetting** — gap below 50% win-rate of the final policy vs early checkpoints.
  Expected: league < mirror (league play should mitigate forgetting).
- **nontransitivity** — spectral measure of payoff-matrix cycling.
  Expected: higher on games where league helps most (Oshi-Zumo, Goofspiel).

If ACH's last-iterate-convergence claim holds, ACH/mirror should approach
PPO/league on the equilibrium metric; league's advantage should show up
primarily in lower forgetting on the cycling games.